In [1]:
# %% [code]
import os
import json
import yaml
import copy
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

# Import your training pipeline and configuration loader from your codebase.
from dendritic_modeling.train_experiments import main as train_main
from dendritic_modeling.config import load_config
from dendritic_modeling.synthetic_datasets import get_unified_datasets
from dendritic_modeling.utils import evaluate_accuracy

# =============================================================================
# Helper functions
# =============================================================================

def load_config_yaml(path):
    with open(path, 'r') as f:
        return yaml.safe_load(f)

def save_config_yaml(cfg_dict, path):
    with open(path, 'w') as f:
        yaml.dump(cfg_dict, f, sort_keys=False)

def parse_experiment_logs(exp_dir):
    """
    Parse param_count and final_loss from logs (or JSON) in exp_dir.
    This function assumes that the training pipeline saves a file named
    'final_results.json' containing fields "param_count" and "final_loss".
    Adapt as needed.
    """
    param_count = None
    final_loss = None
    final_path = os.path.join(exp_dir, "final_results.json")
    if os.path.exists(final_path):
        with open(final_path, 'r') as f:
            data = json.load(f)
        param_count = data.get("param_count", None)
        final_loss = data.get("final_loss", None)
        if final_loss is None:
            final_loss = data.get("test_nll", None)
    return param_count, final_loss

def run_single_experiment(config_base, output_dir, experiment_name):
    """
    Save a temporary config, call the training pipeline (train_main),
    and parse logs for parameter count and final loss.
    
    Returns a dictionary with keys: "param_count", "final_loss", and (dummy) "test_acc".
    (This function does not write aggregated results to disk.)
    """
    os.makedirs(output_dir, exist_ok=True)
    config_path = os.path.join(output_dir, "config.yaml")
    save_config_yaml(config_base, config_path)
    
    try:
        train_main(config_path=config_path, output_dir=output_dir, experiment_name=experiment_name)
    except Exception as e:
        print(f"[ERROR] training failed for {experiment_name}: {e}")
        return None
    
    pcount, loss = parse_experiment_logs(output_dir)
    # For demonstration, we set a dummy test accuracy (adjust if you want to compute it)
    test_acc = 0.0
    return {"param_count": pcount, "final_loss": loss, "test_acc": test_acc}

def run_scaling_experiment(base_config_path, output_base_dir, dim_values, experiment_name_prefix="scaling_exp"):
    """
    For each dimension in dim_values, run two experiments (one with network type "EINet"
    and one with "MLP") while setting "excitatory_branch_factors" = [2, dim].
    
    Returns a dictionary:
       results["EINet"] = [ { "dim": ..., "param_count": ..., "final_loss": ..., ...}, ... ]
       results["MLP"]   = [ { "dim": ..., "param_count": ..., "final_loss": ..., ...}, ... ]
    """
    os.makedirs(output_base_dir, exist_ok=True)
    base_cfg = load_config_yaml(base_config_path)
    results = {"EINet": [], "MLP": []}
    
    for dim in dim_values:
        for net_type in ["EINet", "MLP"]:
            cfg = copy.deepcopy(base_cfg)
            # Set the network type
            cfg["model"]["network"]["type"] = net_type
            # Set excitatory_branch_factors to [2, dim]
            cfg["model"]["network"]["parameters"]["excitatory_branch_factors"] = [2, dim]
            # Optionally, set inhibitory_branch_factors to an empty list to disable inhibition
            cfg["model"]["network"]["parameters"]["inhibitory_branch_factors"] = []
            
            # Create a unique experiment name and output directory
            exp_name = f"{experiment_name_prefix}_{net_type}_dim{dim}"
            exp_dir  = os.path.join(output_base_dir, exp_name)
            print(f"Running experiment {exp_name} ...")
            
            result = run_single_experiment(cfg, exp_dir, exp_name)
            if result is not None:
                result["dim"] = dim
                results[net_type].append(result)
            else:
                print(f"[WARNING] Could not get results for {net_type} dim={dim}")
    
    return results

def plot_loglog(results, output_dir, experiment_name_prefix):
    """
    Create a single plot with two curves (one for EINet and one for MLP):
      x-axis: log(param_count)
      y-axis: log(final_loss)
    """
    plt.figure(figsize=(6,5))
    
    for net_type, color, marker in [("EINet", "blue", "o"), ("MLP", "red", "s")]:
        data_list = results.get(net_type, [])
        # Filter out entries where param_count or final_loss is missing
        data_list = [d for d in data_list if (d["param_count"] is not None and d["final_loss"] is not None)]
        data_list.sort(key=lambda x: x["param_count"])
        
        if len(data_list) == 0:
            continue
        
        x_param = [d["param_count"] for d in data_list]
        y_loss  = [d["final_loss"] for d in data_list]
        
        plt.xscale("log")
        plt.yscale("log")
        plt.plot(x_param, y_loss, marker=marker, color=color, linestyle='-', label=net_type)
    
    plt.xlabel("Param Count (log scale)")
    plt.ylabel("Final Loss (log scale)")
    plt.title(f"Scaling Experiment: {experiment_name_prefix} (EINet vs MLP)")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# =============================================================================
# Experiment setup (modify these parameters as needed)
# =============================================================================
BASE_CONFIG = "../.vscode/config_exp.yaml"  # Path to your base config (adjust if needed)
OUTPUT_DIR  = "results_scaling"             # Base output directory
DIM_LIST    = [2, 4]                        # Example values for the second element of excitatory_branch_factors
EXPERIMENT_NAME = "scaling_exp"

# =============================================================================
# Run the scaling experiments
# =============================================================================
results = run_scaling_experiment(BASE_CONFIG, OUTPUT_DIR, DIM_LIST, experiment_name_prefix=EXPERIMENT_NAME)
print("Aggregated results:", results)

# =============================================================================
# Plot the log-log results
# =============================================================================
plot_loglog(results, OUTPUT_DIR, EXPERIMENT_NAME)
print("[DONE] All scaling experiments complete and plotted.")

ModuleNotFoundError: No module named 'dendritic_modeling.train_experiments'